# Stream Lifecycle and Result Lists

In this lesson, you will learn to trace stream execution, reuse collected results, and create separate editable collection copies.

CSC-239 · Module 10 · Lesson 3 of 4

You can build an ordered map/filter/toList pipeline. Now you will distinguish the pipeline description from its execution and from the list produced by that execution. You will trace reports, repair invalid operations, and build a working copy with boundary tests.

Use the [Module 10 terminology glossary](terms.md) to revisit terms after their explanations in the lesson.


## Learning Goals

- Trace when an ArrayList-backed pipeline observes its source and becomes consumed.
- Distinguish reusing a result from requesting a fresh stream computation, and interpret its count as a long result.
- Create and test an editable shallow collection copy while preserving the original result.


## Why This Matters

A program may describe a report before all the input is ready. After generating the report, another feature may need to read it again, calculate a new total, or edit a working draft. Those requests involve different objects and different rules. A reference to a pipeline does not preserve an earlier report, and another variable name does not make a list editable.

The previous lesson separated an ordered source from its transformed result. This lesson adds the timing and editing decisions that larger programs need. Keeping a pending computation, a completed report, and an editable copy distinct helps preserve information another feature still uses.

Without that distinction, a second computation can target an already used stream, or an edit can target a list that rejects changes. Understanding the contracts lets you choose the correct operation directly and explain the result when testing a program.


## Check Your Starting Point

Identify the source, intermediate operation, and terminal operation in a short pipeline from the previous lesson. Recall the difference between copying a reference and creating a separate collection. Explain why changing a list entry and changing an object referenced by that entry are different actions.


In [ ]:
Your response:

Source, intermediate operation, terminal operation:
Reference assignment versus new collection:
Changing an entry versus changing a referenced object:


<details>
<summary>Show answer</summary>

In a pipeline such as the previous lesson's name report, the source collection supplies entries, mapping and filtering describe intermediate operations, and `toList()` produces the result list. That result can be read without modifying the source collection.

Assigning a reference to another variable gives both variables a way to reach the same object. It does not construct a second collection. Creating a new collection gives it its own list entries, although entries in different lists can refer to the same element object.

Replacing or removing a list entry changes which reference that list stores. Calling a mutating method on a referenced object instead changes that object. Strings are immutable, so the examples here change collection entries without changing String contents in place.

</details>


## Video Demonstration

Watch when the pending pipeline observes its ArrayList source. Then compare the sizes of the report, its editable copy, and the source.

<video controls preload="metadata" width="960">
  <source src="media/03_stream_lifecycle_and_result_lists/demo.mp4" type="video/mp4">
  <track kind="captions" src="media/03_stream_lifecycle_and_result_lists/captions.vtt" srclang="en" label="English">
  Your browser does not support embedded video.
</video>

[Read the stream lifecycle and result lists demonstration transcript](media/03_stream_lifecycle_and_result_lists/transcript.md).


## Concept

### Distinguish describing work from performing it

A supply desk is preparing a report of requested items. Its source ArrayList initially contains `map`. Before producing the report, the desk adds `kit`. Should a pipeline created before that addition include the later request? To answer, we need to distinguish creation of the pipeline from execution of its processing.

**Lazy evaluation** means that the intermediate stages describe work before a terminal operation starts processing the elements. The next example keeps the unfinished pipeline in a variable so those moments appear on separate lines.

```java
Stream<String> pending = source.stream().map(item -> "Item: " + item);
source.add("kit");
List<String> report = pending.toList();
```

`Stream<String>` is the library type for the pipeline's String elements. The complete program imports it from `java.util.stream`. The first statement constructs a pipeline whose mapping rule adds `Item: ` before each processed item. It stores that pipeline in `pending`; it has not yet collected a report.

The next statement adds `kit` to the source ArrayList. Only afterward does `toList()` perform terminal processing and produce `report`. An ArrayList stream is a **late-bound collection source**: in this use, it observes the source contents when terminal processing begins. The report therefore contains `Item: map` and `Item: kit`, in that order.

Creating `pending` did not save a copy of the one-entry collection. However, this example does not permit changes while the pipeline is processing. The addition finishes before the terminal call begins. Different kinds of stream sources can have different binding rules; the behavior demonstrated here is specifically the ArrayList-backed stream.


In [ ]:
import java.util.ArrayList;
import java.util.List;
import java.util.stream.Stream;
ArrayList<String> source = new ArrayList<String>();
source.add("map");
Stream<String> pending = source.stream().map(item -> "Item: " + item);
source.add("kit");
List<String> report = pending.toList();
for (String item : report) {
    System.out.println(item);
}

The complete program prints:

```text
Item: map
Item: kit
```

The first addition happens before the pipeline is created. The second finishes after pipeline creation but before terminal processing. Both entries therefore supply values to the mapping operation when `toList()` collects the report. The loop reads that completed report afterward.

Keep this timing boundary precise: do not add or remove source entries from a lambda running inside the pipeline. Changing the source before processing starts is different from changing it during processing. Next, consider what remains usable after this terminal operation finishes.


### Give each computation a fresh stream

After a report is collected, the source, pipeline, and result have different roles. The source still holds its entries. The result list holds the completed report, which can be read again. The pipeline has already been used.

A **single-use stream** must not be reused for another computation after a terminal operation. The next complete example creates a stream named `pending` over a one-entry source and collects it into `report`. It then deliberately attempts another terminal operation on that same stream:

```java
try {
    pending.count();
} catch (IllegalStateException exception) {
    System.out.println("Create a new stream.");
}
```

`count()` asks for the number of elements. This request fails because `pending` has already been consumed by `toList()`. **IllegalStateException** reports an operation that is invalid for the object's current state. In this example, control moves to the matching catch block, which prints the recovery message. The failed call does not return a count.

The source collection remains available. Calling `source.stream()` requests a new stream for another computation. Reading `report` would instead reuse the already collected result without asking the old pipeline to run again. Before the complete example uses a fresh stream to count, we need to explain the numeric type of that result.


### Interpret a count and its type

The terminal operation **`count()`** returns the number of elements in the stream as a **`long`** value. The Java keyword `long` names a primitive whole-number type with a wider range than `int`. A result has a type even when the program does not first store it in a named variable.

```java
System.out.println("Fresh count: " + source.stream().count());
```

In this statement, `source.stream()` supplies a fresh stream. `count()` produces its whole-number result. Concatenation combines the label with the displayed form of that value, and `println` prints the line. There is no need to convert the count to int just to display it.

The complete example's source contains one `map` entry, so this successful computation prints `Fresh count: 1` after the earlier recovery message. Counting an empty source would return zero. The fresh count concerns the source's entries at that computation; it does not revive the consumed `pending` stream or change the earlier report.


In [ ]:
import java.util.ArrayList;
import java.util.List;
import java.util.stream.Stream;
ArrayList<String> source = new ArrayList<String>();
source.add("map");
Stream<String> pending = source.stream();
List<String> report = pending.toList();
try {
    pending.count();
} catch (IllegalStateException exception) {
    System.out.println("Create a new stream.");
}
System.out.println("Fresh count: " + source.stream().count());

The complete program prints:

```text
Create a new stream.
Fresh count: 1
```

The first terminal operation completes normally. The deliberate `pending.count()` attempt then fails, so it does not produce a numeric result. The matching catch block prints the first line. The final statement creates a new stream and successfully prints its `long` count.

The catch demonstrates the prohibited reuse in this example. It is not the normal way to request another computation: ask the source for a fresh stream directly. Reading the stored `report` again is a different action and remains allowed. A readable result can still have an editing restriction, which the next example tests.


### Handle an unsupported result edit

The previous lesson introduced the unmodifiable list returned by `toList()`. We can now observe its editing restriction directly. The next example collects a source containing only `map`, then tries to add `kit` to the result.

```java
try {
    report.add("kit");
} catch (UnsupportedOperationException exception) {
    System.out.println("Result cannot be edited.");
}
System.out.println("Result size: " + report.size());
```

**UnsupportedOperationException** reports that an object does not support the requested operation. Here, `report.add` cannot add an entry to the unmodifiable result. The catch block prints `Result cannot be edited.` Execution then continues with the size check, which prints `Result size: 1`. The failed edit did not insert kit.

The declared type `List<String>` allows code to name list operations, but it does not promise that every particular list supports editing. The result-producing operation establishes that restriction. Reading the result remains supported; adding, removing, or replacing its entries through this result list is rejected.

This is a different problem from reusing a consumed stream. One failure concerns another computation on an already used pipeline. The other concerns an unsupported change to a completed result list. Their matching exception types let the examples identify which operation failed.


In [ ]:
import java.util.ArrayList;
import java.util.List;
ArrayList<String> source = new ArrayList<String>();
source.add("map");
List<String> report = source.stream().toList();
try {
    report.add("kit");
} catch (UnsupportedOperationException exception) {
    System.out.println("Result cannot be edited.");
}
System.out.println("Result size: " + report.size());

The complete program prints:

```text
Result cannot be edited.
Result size: 1
```

The attempted addition targets `report`, so its unmodifiable contract applies. The matching catch block runs, and the later `size()` call confirms that the report still has one entry. The rejected addition did not partly extend it.

This distinction matters when choosing a working list. A result may support the reads you need while rejecting edits. Rather than assuming that a different variable name changes the object, create a separate collection whose operations fit the task.


### Create a separately editable list

Suppose the desk wants to add a new item to a working list while preserving its completed report. The solution is to construct another collection from the report's current elements.

```java
ArrayList<String> editable = new ArrayList<String>(report);
editable.add("kit");
```

The constructor's argument is the existing report. It creates a new ArrayList containing that report's elements in order. This **collection copy** has its own list structure and supports editing. The second statement adds kit to that new list.

The complete example begins with a one-entry source containing map, then produces a one-entry report and makes the copy. After the addition, it prints `Report: 1`, `Editable: 2`, and `Source: 1`. The sizes differ because only the new ArrayList received the additional entry. Reassigning another variable to the same report would not have created this separate editable structure.

This construction makes a **shallow copy**: it copies element references into a new collection, rather than constructing independent copies of the element objects. The original map String can therefore be referenced by both lists. Separate membership does not imply separate copies of every object.

Strings are immutable, so these examples cannot demonstrate changing shared element contents in place. They demonstrate separate collection membership and shared references. If elements were mutable objects, modifying one shared object could be visible through both collections. Creating this ArrayList alone would not provide a deep copy of those objects.


In [ ]:
import java.util.ArrayList;
import java.util.List;
ArrayList<String> source = new ArrayList<String>();
source.add("map");
List<String> report = source.stream().toList();
ArrayList<String> editable = new ArrayList<String>(report);
editable.add("kit");
System.out.println("Report: " + report.size());
System.out.println("Editable: " + editable.size());
System.out.println("Source: " + source.size());

The complete program prints:

```text
Report: 1
Editable: 2
Source: 1
```

The report is collected before the copy is constructed. The constructor gives `editable` its own list structure; adding kit changes only that structure. Neither the source nor the original report receives the new entry.

Keep the object roles distinct:

| Object | What it represents | Appropriate next action |
| --- | --- | --- |
| Source ArrayList | Stored input entries | Request a fresh stream for another computation |
| Pending Stream | One computation not yet completed | Invoke one terminal operation |
| Result List | Entries produced by a completed computation | Read again or copy into an editable collection |
| Editable ArrayList copy | Separate list structure initialized from the result | Add, remove, or replace its own entries |

An empty source can still produce an empty result and an empty copy. Their membership is separate even when all sizes are zero. Use returned values and completed results as evidence. Printing inside an intermediate lambda does not establish an exact execution schedule; the implementation may avoid work that cannot affect the terminal result.


## Worked Example

A supply desk starts with one requested item, map. A second request, kit, arrives before the report is collected. Each report entry adds `Item: ` before the item name. The desk then needs a working copy that includes an extra pen entry while preserving the completed report. All four printed measurements count entries, not characters.

**Prepare the description and finish the input.** The first part separates pipeline creation from execution:

```java
source.add("map");
Stream<String> pending = source.stream().map(item -> "Item: " + item);
source.add("kit");
```

The first line adds the initial request. The next stores the mapping pipeline in `pending`; it has not collected a result. The last line finishes the second source addition before the terminal operation begins. The ArrayList source will therefore supply both requests to this computation.

**Collect one report and read its size.**

```java
List<String> report = pending.toList();
System.out.println("Reported: " + report.size());
```

The terminal operation consumes `pending` and produces the list assigned to `report`. The following statement reads the completed list's size. It does not ask the old stream to run again.

**Give the draft its own list structure.**

```java
ArrayList<String> editable = new ArrayList<String>(report);
editable.add("Item: pen");
```

The constructor uses the report's current entries to initialize a new ArrayList. The addition then targets that new list. The pen is already a report label, so this step adds the full text `Item: pen`; it does not send another source value through the consumed mapping pipeline.

The remaining statements read the original result and the editable copy separately. The final `source.stream().count()` requests a fresh computation over the source. Assemble those steps in the complete program below.


In [ ]:
import java.util.ArrayList;
import java.util.List;
import java.util.stream.Stream;
ArrayList<String> source = new ArrayList<String>();
source.add("map");
Stream<String> pending = source.stream().map(item -> "Item: " + item);
source.add("kit");
List<String> report = pending.toList();
System.out.println("Reported: " + report.size());
ArrayList<String> editable = new ArrayList<String>(report);
editable.add("Item: pen");
System.out.println("Original result: " + report.size());
System.out.println("Editable copy: " + editable.size());
System.out.println("Fresh count: " + source.stream().count());


Expected output:

```text
Reported: 2
Original result: 2
Editable copy: 3
Fresh count: 2
```

The ArrayList contains map and kit when toList begins. The report therefore contains two entries. Adding a pen entry to a new ArrayList changes only that copy, so the report stays at two and the copy grows to three. A fresh stream counts the two source entries.

The source remains available for the fresh count, and the report remains available for later reads. Neither is the same object as the new editable list. The copy initially shares element references with the report; adding pen changes only its membership.


<details id="animation-pending-source-terminal-timeline" class="animation-panel" open>
<summary>Trace collection timing and a separate editable copy — show or hide animation</summary>
<p><img src="media/03_stream_lifecycle_and_result_lists/pending-source-terminal-timeline.gif" alt="The source contains map when pending is created. Kit is added before collection. The completed report contains Item: map and Item: kit. A separate copy gains Item: pen while the report keeps two entries. A fresh stream counts the two source entries." width="960" style="max-width:100%;height:auto;"></p>
</details>

The first phases distinguish stored input from pending work. Collection reads map and kit because both additions finish before terminal processing begins. The later copy has its own list structure, so adding the pen label changes only that copy. The final count requests a fresh stream over the source. This silent loop lasts 25 seconds. Hiding it removes the visible motion. [View the final state as a still image](media/03_stream_lifecycle_and_result_lists/pending-source-terminal-timeline_still.png).


## Guided Practice

Use the distinctions you just traced with a different input. Predict before each run, keep the prediction when recording actual results, and use the hidden explanation only after writing your reasoning.


### Predict the report, copy, and fresh count

The complete program below starts with badge, creates a pending mapping pipeline, and then adds card and key before collection. Predict all four printed lines before running it. Identify which source entries exist when `toList()` starts and which list receives the pen entry. Explain whether creating `pending` immediately freezes the input and whether the final count uses that stream again.


In [ ]:
Your response:

Four predicted lines:
Source entries when collection starts:
List receiving pen:
Does pending freeze the input?
Does the fresh count reuse pending?


In [ ]:
import java.util.ArrayList;
import java.util.List;
import java.util.stream.Stream;
ArrayList<String> source = new ArrayList<String>();
source.add("badge");
Stream<String> pending = source.stream().map(item -> "Item: " + item);
source.add("card");
source.add("key");
List<String> report = pending.toList();
System.out.println("Reported: " + report.size());
ArrayList<String> editable = new ArrayList<String>(report);
editable.add("Item: pen");
System.out.println("Original result: " + report.size());
System.out.println("Editable copy: " + editable.size());
System.out.println("Fresh count: " + source.stream().count());


Run the complete prediction program in the Workspace. Keep your original prediction and record every actual output line. Explain each confirmed or corrected result, including why the source count can differ from the copy's size.

Trace these five points: after creating `pending`, just before `toList()`, just after `toList()`, after creating `editable`, and after adding pen. At each point, record the source contents and the report and copy contents when they exist. Identify the terminal operation that consumes `pending`.

State what can be read again, what needs a fresh stream, and what the collection constructor does and does not copy. Name the type returned by the fresh count and explain why printing it does not require conversion to int.


In [ ]:
Your response:

Actual four lines and corrections:
After pending creation: source / report / copy:
Before toList: source / report / copy:
After toList: source / report / copy:
After copy creation: source / report / copy:
After pen addition: source / report / copy:
Terminal operation consuming pending:
Reusable result versus fresh computation:
What the constructor copies:
Count result type and direct printing:


<details>
<summary>Show answer</summary>

Creating pending describes the mapping without collecting a report. The ArrayList holds badge, card and key when toList begins, so the report contains Item: badge, Item: card and Item: key. Reported is three. The ArrayList constructor copies those entries into a separate editable list. Adding Item: pen changes only that copy, leaving the report size three and making the copy size four. The final count uses a fresh stream over the original three source entries and reports three. count returns long; printing it does not require changing it to int. Before toList, pending refers to the uncompleted computation. After toList, that stream has been used, while report holds a list that can be read again. The collection constructor creates a separate list structure but initially copies the same String references; it does not create new String objects. This is the shallow copy described in the reading. The demonstrated source timing belongs to this ArrayList source, with changes before or after terminal processing rather than inside it.

```java
import java.util.ArrayList;
import java.util.List;
import java.util.stream.Stream;
ArrayList<String> source = new ArrayList<String>();
source.add("badge");
Stream<String> pending = source.stream().map(item -> "Item: " + item);
source.add("card");
source.add("key");
List<String> report = pending.toList();
System.out.println("Reported: " + report.size());
ArrayList<String> editable = new ArrayList<String>(report);
editable.add("Item: pen");
System.out.println("Original result: " + report.size());
System.out.println("Editable copy: " + editable.size());
System.out.println("Fresh count: " + source.stream().count());
```

Expected output:

```text
Reported: 3
Original result: 3
Editable copy: 4
Fresh count: 3
```

Common error: Counting only the entry present when pending was created. Adding the copy’s new entry to the report or source count. Treating a fresh stream as another use of the old stream.

</details>


### Read a report again and request a fresh computation

The next complete program uses oak and elm to produce Tree labels. It then deliberately attempts a second terminal operation on the consumed stream, with a catch block that allows execution to continue. Predict every output line before running. Distinguish `pending.count()`, the loop over `report`, `report.get(0)`, and the new stream used for the final count.


In [ ]:
Your response:

Predicted output lines:
Purpose of old stream call, report reads, and fresh count:


In [ ]:
import java.util.ArrayList;
import java.util.List;
import java.util.stream.Stream;
ArrayList<String> source = new ArrayList<String>();
source.add("oak");
Stream<String> pending = source.stream().map(item -> "Tree: " + item);
source.add("elm");
List<String> report = pending.toList();
System.out.println("First report: " + report.size());
try {
    pending.count();
} catch (IllegalStateException problem) {
    System.out.println("Fresh stream needed.");
}
for (String item : report) {
    System.out.println(item);
}
System.out.println("Read again: " + report.get(0));
System.out.println("Fresh count: " + source.stream().count());


Run the complete program. Keep your prediction and record every actual line, explaining any correction. Identify the rejected call and its exception type. Explain why the loop and `get` can read the stored result again, and why the final count succeeds. Distinguish demonstrating invalid reuse with a catch from the normal choice of requesting a fresh stream.


In [ ]:
Your response:

Actual lines and corrections:
Rejected operation and exception:
Why report rereads succeed:
Why fresh count succeeds:
Normal way to request another computation:


<details>
<summary>Show answer</summary>

The pending stream is collected once, producing the two Tree entries. Calling count on that same stream is rejected with IllegalStateException and the catch block prints Fresh stream needed. The completed report remains usable: the loop reads both entries and get reads its first entry again. The final count succeeds because source.stream() creates a new stream. The caught error demonstrates the invalid reuse; requesting a fresh stream directly is the normal way to perform another computation.

```java
import java.util.ArrayList;
import java.util.List;
import java.util.stream.Stream;
ArrayList<String> source = new ArrayList<String>();
source.add("oak");
Stream<String> pending = source.stream().map(item -> "Tree: " + item);
source.add("elm");
List<String> report = pending.toList();
System.out.println("First report: " + report.size());
try {
    pending.count();
} catch (IllegalStateException problem) {
    System.out.println("Fresh stream needed.");
}
for (String item : report) {
    System.out.println(item);
}
System.out.println("Read again: " + report.get(0));
System.out.println("Fresh count: " + source.stream().count());
```

Expected output:

```text
First report: 2
Fresh stream needed.
Tree: oak
Tree: elm
Read again: Tree: oak
Fresh count: 2
```

Common error: Expecting the consumed stream to restart automatically. Treating a list read as another terminal operation on pending. Using the catch block as the normal method for obtaining another result.

<details id="animation-consumed-stream-reusable-report" class="animation-panel" open>
<summary>Separate a consumed stream from a reusable report — show or hide animation</summary>
<p><img src="media/03_stream_lifecycle_and_result_lists/consumed-stream-reusable-report.gif" alt="The report contains Tree: oak and Tree: elm. Reusing pending for count throws IllegalStateException, and its handler prints Fresh stream needed. The stored report is read in a loop and again with get. A fresh source stream counts two entries." width="960" style="max-width:100%;height:auto;"></p>
</details>

The rejected count and its handler are separate phases. After the handler, the loop and get call read the completed report without using pending again. The last count succeeds because source.stream() supplies a fresh stream. These reads and the fresh computation have different targets. This silent loop lasts 19 seconds. Hiding it removes the visible motion. [View the final state as a still image](media/03_stream_lifecycle_and_result_lists/consumed-stream-reusable-report_still.png).

</details>


### Compare rejected result edits with an editable copy

The next complete program creates a two-entry Tree report, attempts to remove its first entry, and catches the result's unsupported operation. It later makes an ArrayList copy and removes the first entry of that copy. Predict every printed line before running, and explain why the two removal attempts can behave differently.


In [ ]:
Your response:

Remove-case predicted lines:
Why the two removals can differ:


In [ ]:
import java.util.ArrayList;
import java.util.List;
ArrayList<String> source = new ArrayList<String>();
source.add("oak");
source.add("elm");
List<String> report = source.stream().map(item -> "Tree: " + item).toList();
try {
    report.remove(0);
} catch (UnsupportedOperationException problem) {
    System.out.println("Copy before editing.");
}
System.out.println("Report first: " + report.get(0));
ArrayList<String> editable = new ArrayList<String>(report);
editable.remove(0);
System.out.println("Copy first: " + editable.get(0));
System.out.println("Report size: " + report.size());
System.out.println("Copy size: " + editable.size());


Run the complete remove-case program and record its output alongside your prediction. Explain what the catch block tells you, whether the report was partly changed, and which entries and sizes remain after the copy's removal. Explain why simply changing the declared reference type would not change the report's editing contract.


In [ ]:
Your response:

Actual remove-case lines and corrections:
Caught failure and unchanged report:
Report and copy entries/sizes:
Why declared reference type does not grant edit support:


Keep the source, copy construction, and `editable.remove(0)` unchanged. Plan two separate complete variants: replace only `report.remove(0)` with `report.add("Tree: ash")`, then with `report.set(0, "Tree: ash")`. Predict all output lines for each case before editing or running either one. State which collection receives the attempted change.


In [ ]:
Your response:

Add-case predicted lines:
Set-case predicted lines:
Target collection for each attempt:


Edit the existing complete support program and run each planned variant separately. Record the actual output for each, correct your predictions where needed, and explain whether the report was partly changed. Restore `report.remove(0)` and rerun the baseline. Explain what these checks establish about report edits and copy edits.


In [ ]:
Your response:

Actual add-case output and explanation:
Actual set-case output and explanation:
Prediction corrections:
Restored remove-case output:
What the comparisons establish:


<details>
<summary>Show answer</summary>

The remove call on report is rejected with UnsupportedOperationException, so the catch block prints Copy before editing. The report still begins with Tree: oak. A new ArrayList holds a separate copy of the report’s entries. Removing its first entry leaves Tree: elm in the copy while the original report stays at size two. The copy has size one. The add and set comparison programs are also rejected when they target report; each then performs the same successful removal on a newly created editable copy. Changing the declared reference type alone would not make the original list support edits.

```java
import java.util.ArrayList;
import java.util.List;
ArrayList<String> source = new ArrayList<String>();
source.add("oak");
source.add("elm");
List<String> report = source.stream().map(item -> "Tree: " + item).toList();
try {
    report.remove(0);
} catch (UnsupportedOperationException problem) {
    System.out.println("Copy before editing.");
}
System.out.println("Report first: " + report.get(0));
ArrayList<String> editable = new ArrayList<String>(report);
editable.remove(0);
System.out.println("Copy first: " + editable.get(0));
System.out.println("Report size: " + report.size());
System.out.println("Copy size: " + editable.size());
```

Expected output:

```text
Copy before editing.
Report first: Tree: oak
Copy first: Tree: elm
Report size: 2
Copy size: 1
```

Common error: Assuming List guarantees that every implementation supports edits. Assuming a caught rejected edit partially changed the result. Assigning another reference to report instead of constructing a new list.

**Check case 2.** The attempted addition is rejected before an entry is added to report. The later ArrayList copy is still made from the two original entries, and its removal leaves Tree: elm.

```java
import java.util.ArrayList;
import java.util.List;
ArrayList<String> source = new ArrayList<String>();
source.add("oak");
source.add("elm");
List<String> report = source.stream().map(item -> "Tree: " + item).toList();
try {
    report.add("Tree: ash");
} catch (UnsupportedOperationException problem) {
    System.out.println("Copy before editing.");
}
System.out.println("Report first: " + report.get(0));
ArrayList<String> editable = new ArrayList<String>(report);
editable.remove(0);
System.out.println("Copy first: " + editable.get(0));
System.out.println("Report size: " + report.size());
System.out.println("Copy size: " + editable.size());
```

Expected output:

```text
Copy before editing.
Report first: Tree: oak
Copy first: Tree: elm
Report size: 2
Copy size: 1
```

**Check case 3.** The attempted replacement is rejected, so report still starts with Tree: oak. The independent editable copy accepts removal and retains only Tree: elm.

```java
import java.util.ArrayList;
import java.util.List;
ArrayList<String> source = new ArrayList<String>();
source.add("oak");
source.add("elm");
List<String> report = source.stream().map(item -> "Tree: " + item).toList();
try {
    report.set(0, "Tree: ash");
} catch (UnsupportedOperationException problem) {
    System.out.println("Copy before editing.");
}
System.out.println("Report first: " + report.get(0));
ArrayList<String> editable = new ArrayList<String>(report);
editable.remove(0);
System.out.println("Copy first: " + editable.get(0));
System.out.println("Report size: " + report.size());
System.out.println("Copy size: " + editable.size());
```

Expected output:

```text
Copy before editing.
Report first: Tree: oak
Copy first: Tree: elm
Report size: 2
Copy size: 1
```

The animation below returns to the complete remove-case baseline.

<details id="animation-unmodifiable-result-and-shallow-copy" class="animation-panel" open>
<summary>Follow a rejected report edit and an allowed copy edit — show or hide animation</summary>
<p><img src="media/03_stream_lifecycle_and_result_lists/unmodifiable-result-and-shallow-copy.gif" alt="Reference keys A and B identify Tree: oak and Tree: elm. The report rejects removal with UnsupportedOperationException and keeps both entries. A new ArrayList initially holds the same references in a separate list structure. Removing A only from the copy leaves report size two and copy size one." width="960" style="max-width:100%;height:auto;"></p>
</details>

The rejected report removal changes no entry. The new ArrayList has separate list membership while initially referring to the same String elements, labeled A and B in the diagram. Removing the first copy entry changes only that membership; no String contents change. This silent loop lasts 22 seconds. Hiding it removes the visible motion. [View the final state as a still image](media/03_stream_lifecycle_and_result_lists/unmodifiable-result-and-shallow-copy_still.png).

</details>


### Complete the stream and copy choices

Replace PENDING_TYPE, FINISH, COPY_FROM and NEW_STREAM in the displayed program. Choose the pending stream type, the operation that produces the report, the collection passed to the copy constructor, and the method that creates a fresh stream for counting. Use `Stream<String>`, toList, report and stream in the matching places. Before completing the program, state your four replacements, their reasons, and the expected four output lines. Then put the complete finished program in the work cell and run it.

This intentionally incomplete sample is for completion:

```java
import java.util.ArrayList;
import java.util.List;
import java.util.stream.Stream;
ArrayList<String> source = new ArrayList<String>();
source.add("badge");
PENDING_TYPE pending = source.stream().map(item -> "Item: " + item);
source.add("card");
source.add("key");
List<String> report = pending.FINISH();
System.out.println("Reported: " + report.size());
ArrayList<String> editable = new ArrayList<String>(COPY_FROM);
editable.add("Item: pen");
System.out.println("Original result: " + report.size());
System.out.println("Editable copy: " + editable.size());
System.out.println("Fresh count: " + source.NEW_STREAM().count());
```


In [ ]:
Your response:

Four replacements and reasons:
Predicted four output lines:


Record the actual output of your completed program. Explain why each of the four replacements matches its object's role. In particular, distinguish the collection passed to the copy constructor from the collection asked to provide a fresh stream for counting.


In [ ]:
Your response:

Actual output:
Why each replacement fits its role:
Copy argument versus fresh-stream source:


<details>
<summary>Show answer</summary>

Use `Stream<String>` for PENDING_TYPE, toList for FINISH, report for COPY_FROM and stream for NEW_STREAM. These choices preserve the difference between the pending computation, its completed list and a new list structure for edits. The last expression asks the source for a new stream rather than invoking another terminal operation on pending. Creating pending describes the mapping without collecting a report. The ArrayList holds badge, card and key when toList begins, so the report contains Item: badge, Item: card and Item: key. Reported is three. The ArrayList constructor copies those entries into a separate editable list. Adding Item: pen changes only that copy, leaving the report size three and making the copy size four. The final count uses a fresh stream over the original three source entries and reports three. count returns long; printing it does not require changing it to int.

```java
import java.util.ArrayList;
import java.util.List;
import java.util.stream.Stream;
ArrayList<String> source = new ArrayList<String>();
source.add("badge");
Stream<String> pending = source.stream().map(item -> "Item: " + item);
source.add("card");
source.add("key");
List<String> report = pending.toList();
System.out.println("Reported: " + report.size());
ArrayList<String> editable = new ArrayList<String>(report);
editable.add("Item: pen");
System.out.println("Original result: " + report.size());
System.out.println("Editable copy: " + editable.size());
System.out.println("Fresh count: " + source.stream().count());
```

Expected output:

```text
Reported: 3
Original result: 3
Editable copy: 4
Fresh count: 3
```

Common error: Using a list type for the pending computation. Copying source when the task needs mapped report entries. Trying to use pending again for the final count.

</details>


### Move a source change after collection

In the complete baseline below, move only `source.add("key")` so that it runs immediately after the assignment to `report` and before the Reported print. Keep the mapping, copy edit, and fresh count unchanged. Predict the four output lines before editing or running. Explain which collections can now include key and why the timing of its addition matters.


In [ ]:
Your response:

Predicted key-after-collection lines:
Collections that can include key and timing reason:


In [ ]:
import java.util.ArrayList;
import java.util.List;
import java.util.stream.Stream;
ArrayList<String> source = new ArrayList<String>();
source.add("badge");
Stream<String> pending = source.stream().map(item -> "Item: " + item);
source.add("card");
source.add("key");
List<String> report = pending.toList();
System.out.println("Reported: " + report.size());
ArrayList<String> editable = new ArrayList<String>(report);
editable.add("Item: pen");
System.out.println("Original result: " + report.size());
System.out.println("Editable copy: " + editable.size());
System.out.println("Fresh count: " + source.stream().count());


Run the complete program with the key addition moved. Record all four actual lines and explain any correction to your prediction. Compare the source at terminal processing with the source at the final fresh count. Explain why the completed report does not acquire the later source addition.


In [ ]:
Your response:

Actual key-after-collection lines and corrections:
Source at collection and at count:
Why the completed report stays unchanged:


Now move the card addition after collection too, placing it before the moved key addition. Keep the other statements unchanged. Predict the four output lines before making and running this second change, and identify the source entries present at the moment of collection.


In [ ]:
Your response:

Predicted card-and-key-after-collection lines:
Source entries present at collection:


Run the second complete variant and record the actual lines. Explain why the report and source can now have different sizes. These additions finish after collection; they do not run inside the pipeline. Restore both card and key additions to their original positions before `toList()`, rerun, and record the restored baseline output.


In [ ]:
Your response:

Actual second-variant lines and explanation:
Restored baseline output:


<details>
<summary>Show answer</summary>

With key added after toList, terminal processing observes badge and card only. The report therefore has two entries; the copy grows from two to three when Item: pen is added. The source later contains all three entries, so the fresh count is three. Moving card after collection as well leaves only badge in the generated report. That report stays at one, its copy grows to two, and the source still reaches three. These changes occur after processing, not during it.

```java
import java.util.ArrayList;
import java.util.List;
import java.util.stream.Stream;
ArrayList<String> source = new ArrayList<String>();
source.add("badge");
Stream<String> pending = source.stream().map(item -> "Item: " + item);
source.add("card");
List<String> report = pending.toList();
source.add("key");
System.out.println("Reported: " + report.size());
ArrayList<String> editable = new ArrayList<String>(report);
editable.add("Item: pen");
System.out.println("Original result: " + report.size());
System.out.println("Editable copy: " + editable.size());
System.out.println("Fresh count: " + source.stream().count());
```

Expected output:

```text
Reported: 2
Original result: 2
Editable copy: 3
Fresh count: 3
```

Common error: Assuming a previously produced report automatically receives later source entries. Moving the source edits into the mapping lambda. Changing the fresh count to reuse the consumed stream.

**Additional test: `Both card and key are added after collection`.** Only badge is present when toList starts. The report contains one mapped entry; adding Item: pen grows the copy to two. Both later source additions are included in the fresh count of three.

```java
import java.util.ArrayList;
import java.util.List;
import java.util.stream.Stream;
ArrayList<String> source = new ArrayList<String>();
source.add("badge");
Stream<String> pending = source.stream().map(item -> "Item: " + item);
List<String> report = pending.toList();
source.add("card");
source.add("key");
System.out.println("Reported: " + report.size());
ArrayList<String> editable = new ArrayList<String>(report);
editable.add("Item: pen");
System.out.println("Original result: " + report.size());
System.out.println("Editable copy: " + editable.size());
System.out.println("Fresh count: " + source.stream().count());
```

Expected output:

```text
Reported: 1
Original result: 1
Editable copy: 2
Fresh count: 3
```

</details>


### Repair an alias that cannot accept edits

The displayed program intends `editable` to be an independent list that accepts changes. Identify the declaration that instead creates another reference to `report`. Predict the output before failure, identify the rejected operation and exception type, and state which later print statements will not run.

Repair that declaration with a suitable ArrayList constructor while keeping the source, mapping, and print statements unchanged. Put only the complete repaired program in the Java work cell and run it.

This sample is intentionally faulty and is provided for diagnosis:

```java
import java.util.ArrayList;
import java.util.List;
import java.util.stream.Stream;
ArrayList<String> source = new ArrayList<String>();
source.add("badge");
Stream<String> pending = source.stream().map(item -> "Item: " + item);
source.add("card");
source.add("key");
List<String> report = pending.toList();
System.out.println("Reported: " + report.size());
List<String> editable = report;
editable.add("Item: pen");
System.out.println("Original result: " + report.size());
System.out.println("Editable copy: " + editable.size());
System.out.println("Fresh count: " + source.stream().count());
```


In [ ]:
Your response:

Faulty declaration:
Predicted output before failure:
Rejected operation and exception:
Later statements not reached:
Planned repaired declaration:


Record your repaired declaration and every actual output line. Explain why adding to the repaired list succeeds while the original report stays unchanged. Distinguish constructing a second list from giving the original list another reference name.


In [ ]:
Your response:

Repaired declaration:
Actual repaired output:
Why independent copy edits succeed:


After the repaired copy's pen addition, insert `editable.set(0, "Item: changed")`. After the count, print `report.get(0)` and `editable.get(0)` with the labels `"Original first: "` and `"Copy first: "`. Predict all output lines and both first entries before editing or running this complete variant.


In [ ]:
Your response:

Predicted output including both first-entry checks:
Expected report and copy first entries:


Run the complete first-entry comparison and record its output. Explain why replacing a copy entry does not replace the report's entry or modify a shared String object. Restore the repaired baseline by removing the added replacement and two first-entry print statements, then rerun and record its output.


In [ ]:
Your response:

Actual first-entry comparison output:
Why replacing an entry does not mutate the shared String:
Restored repaired-baseline output:


<details>
<summary>Show answer</summary>

The faulty assignment makes editable refer to the same unmodifiable list as report. The program prints `Reported: 3`, then `editable.add("Item: pen")` throws `UnsupportedOperationException`. The three later count lines are not reached. A new variable name does not create a new list or change its editing rules. The repair uses `new ArrayList<String>(report)` to create a separate list structure. Adding an entry to that copy succeeds and leaves the report unchanged. Replacing index zero in the repaired copy also changes only that copy’s entry: report still begins with Item: badge while the copy begins with Item: changed. The constructor initially copies element references, but set replaces one list entry rather than changing the shared String object.

```java
import java.util.ArrayList;
import java.util.List;
import java.util.stream.Stream;
ArrayList<String> source = new ArrayList<String>();
source.add("badge");
Stream<String> pending = source.stream().map(item -> "Item: " + item);
source.add("card");
source.add("key");
List<String> report = pending.toList();
System.out.println("Reported: " + report.size());
ArrayList<String> editable = new ArrayList<String>(report);
editable.add("Item: pen");
System.out.println("Original result: " + report.size());
System.out.println("Editable copy: " + editable.size());
System.out.println("Fresh count: " + source.stream().count());
```

Expected output:

```text
Reported: 3
Original result: 3
Editable copy: 4
Fresh count: 3
```

Common error: Changing the reference type without creating a new collection. Editing report when only the working copy should change. Claiming the constructor makes new copies of all element objects.

**Additional test: `Replace the first entry only in the repaired copy`.** The original report retains Item: badge at index zero while the copy now holds Item: changed there. The earlier added pen still makes the copy size four. Replacing a copy entry does not change source or report entries.

```java
import java.util.ArrayList;
import java.util.List;
import java.util.stream.Stream;
ArrayList<String> source = new ArrayList<String>();
source.add("badge");
Stream<String> pending = source.stream().map(item -> "Item: " + item);
source.add("card");
source.add("key");
List<String> report = pending.toList();
System.out.println("Reported: " + report.size());
ArrayList<String> editable = new ArrayList<String>(report);
editable.add("Item: pen");
editable.set(0, "Item: changed");
System.out.println("Original result: " + report.size());
System.out.println("Editable copy: " + editable.size());
System.out.println("Fresh count: " + source.stream().count());
System.out.println("Original first: " + report.get(0));
System.out.println("Copy first: " + editable.get(0));
```

Expected output:

```text
Reported: 3
Original result: 3
Editable copy: 4
Fresh count: 3
Original first: Item: badge
Copy first: Item: changed
```

</details>


## Independent Practice

### Build an editable room report

Create a complete program for a room-report task. Start an ArrayList named `source` with lab. Build a pending stream that maps each room to `"Room: "` followed by its name, then add hall and desk to the source before terminal execution. Collect `report` with `toList()`.

Copy `report` into an ArrayList named `editable`. Remove index zero from the copy, then print its remaining entries in order. Print `"Original result: "` with the report size, `"Editable copy: "` with the copy size, and `"Fresh count: "` with a new stream's source count. Include all three imports and complete setup.

The required baseline prints Room: hall, Room: desk, Original result: 3, Editable copy: 2, and Fresh count: 3 on separate lines. Before coding, plan the creation, collection, copying, removal, and counting steps. Explain why the removal belongs on the copy and identify the source entries that will exist when `toList()` starts.


In [ ]:
Your response:

Planned creation / collection / copy / removal / fresh count:
Expected baseline lines:
Source entries at collection:
Why removal belongs on the copy:


Run your complete room program and record every actual baseline output line. Explain why the copy can change while the report remains readable and unchanged, and why the count requests a fresh stream. Name the type returned by that count and distinguish the source count from the copy's size.


In [ ]:
Your response:

Actual baseline lines:
Separate copy and unchanged report explanation:
Fresh count reason and result type:
Source count versus copy size:


### Check empty and one-entry sources

For the boundary tests, replace the copy's removal with an `if` statement that calls `remove(0)` only when `editable.size()` is greater than zero. Keep the pipeline and result-reading statements in each complete program.

Predict every output line for all three cases before running any variant:

1. A source with no room additions.
2. A source containing only lab.
3. The empty source again, with `"Room: studio"` added to `editable` after the guarded removal but before printing.

Also predict the restored original three-room baseline. Explain why an empty copy cannot remove index zero, and why adding to that copy should leave the original source and report empty.


In [ ]:
Your response:

Empty-source predicted output:
One-room predicted output:
Empty-copy addition predicted output:
Restored baseline predicted output:
Empty-index and separate-membership reasoning:


Edit and run the complete program for each planned case. Record the actual output and an explanation for each, including any prediction correction. Compare the source, report, and copy sizes: a zero-entry copy can result from an empty source or from removing the only copied room, and those situations have different original counts.

Restore the original three-room baseline and rerun it. Record its output and explain what these tests show about valid indices, independent membership, and requesting fresh computations.


In [ ]:
Your response:

Actual empty-source output and explanation:
Actual one-room output and explanation:
Actual empty-copy addition output and explanation:
Corrections and count comparisons:
Restored baseline output:
What the tests establish:


<details>
<summary>Show answer</summary>

The ArrayList contains lab, hall and desk before toList starts. Mapping produces Room: lab, Room: hall and Room: desk in that order. The new ArrayList copies the three result entries into a separate editable list. Removing index zero from the copy leaves Room: hall and Room: desk to print. The report still has three entries, the copy has two, and a fresh stream counts three source entries. The original report is unmodifiable, so the removal belongs on the copy. Copying the list structure initially copies element references; these String values are not changed by the removal. For an empty source, toList and the copy constructor produce empty lists. The guard skips removal because index zero is absent; all three sizes are zero. With only lab, the report has one entry and the copy’s allowed removal leaves it empty. Adding Room: studio to a copy made from an empty report creates one copy entry while source and report remain empty. These tests change the supplied input or copy while retaining a fresh stream for each new count.

```java
import java.util.ArrayList;
import java.util.List;
import java.util.stream.Stream;
ArrayList<String> source = new ArrayList<String>();
source.add("lab");
Stream<String> pending = source.stream().map(room -> "Room: " + room);
source.add("hall");
source.add("desk");
List<String> report = pending.toList();
ArrayList<String> editable = new ArrayList<String>(report);
editable.remove(0);
for (String room : editable) {
    System.out.println(room);
}
System.out.println("Original result: " + report.size());
System.out.println("Editable copy: " + editable.size());
System.out.println("Fresh count: " + source.stream().count());
```

Expected output:

```text
Room: hall
Room: desk
Original result: 3
Editable copy: 2
Fresh count: 3
```

Common error: Collecting before adding the required hall and desk entries. Removing from the unmodifiable report. Reusing the consumed pending stream for the count.

**Additional test: Empty source with guarded removal.** There are no source additions. Both produced lists are empty. The guard skips removal and the printing loop has no entries, but all three zero-size lines still run.

```java
import java.util.ArrayList;
import java.util.List;
import java.util.stream.Stream;
ArrayList<String> source = new ArrayList<String>();
Stream<String> pending = source.stream().map(room -> "Room: " + room);
List<String> report = pending.toList();
ArrayList<String> editable = new ArrayList<String>(report);
if (editable.size() > 0) {
    editable.remove(0);
}
for (String room : editable) {
    System.out.println(room);
}
System.out.println("Original result: " + report.size());
System.out.println("Editable copy: " + editable.size());
System.out.println("Fresh count: " + source.stream().count());
```

Expected output:

```text
Original result: 0
Editable copy: 0
Fresh count: 0
```

**Additional test: Only lab with guarded removal.** The source and report contain one room. Removing the only copied entry is valid and leaves no copy entry to print. The fresh source count remains one.

```java
import java.util.ArrayList;
import java.util.List;
import java.util.stream.Stream;
ArrayList<String> source = new ArrayList<String>();
source.add("lab");
Stream<String> pending = source.stream().map(room -> "Room: " + room);
List<String> report = pending.toList();
ArrayList<String> editable = new ArrayList<String>(report);
if (editable.size() > 0) {
    editable.remove(0);
}
for (String room : editable) {
    System.out.println(room);
}
System.out.println("Original result: " + report.size());
System.out.println("Editable copy: " + editable.size());
System.out.println("Fresh count: " + source.stream().count());
```

Expected output:

```text
Original result: 1
Editable copy: 0
Fresh count: 1
```

**Additional test: Add a room only to the initially empty copy.** The guarded empty removal does nothing. Adding Room: studio afterward grows only the independent copy, so its one room is printed while source and original result stay at size zero.

```java
import java.util.ArrayList;
import java.util.List;
import java.util.stream.Stream;
ArrayList<String> source = new ArrayList<String>();
Stream<String> pending = source.stream().map(room -> "Room: " + room);
List<String> report = pending.toList();
ArrayList<String> editable = new ArrayList<String>(report);
if (editable.size() > 0) {
    editable.remove(0);
}
editable.add("Room: studio");
for (String room : editable) {
    System.out.println(room);
}
System.out.println("Original result: " + report.size());
System.out.println("Editable copy: " + editable.size());
System.out.println("Fresh count: " + source.stream().count());
```

Expected output:

```text
Room: studio
Original result: 0
Editable copy: 1
Fresh count: 0
```

</details>


## Summary

A pending pipeline describes work. A terminal operation performs its computation, and that stream must not be reused. In the demonstrated ArrayList source, changes completed before terminal processing differ from additions after the report is collected. Neither example permits interference during processing.

A completed `toList()` result supports repeated reads and rejects entry edits. A new ArrayList gives an editable copy its own list structure while initially retaining references to the same element objects. A fresh stream computes again from the source; it does not reset the old stream or change an earlier report.

Close the answers and explain why the source, stream, result, and editable copy have different appropriate next actions. Distinguish rereading a report from generating a new one.


In [ ]:
Your response:

Source next action:
Pending stream next action and after-use restriction:
Completed report next action:
Editable copy next action:
Rereading versus generating a new report:


<details>
<summary>Show answer</summary>

The source holds the input entries and can provide a fresh stream for another computation. A pending stream describes work that can be completed by a terminal operation; after use, saving its reference does not reset it.

The completed report stores the result entries and supports repeated reads. Its unmodifiable contract rejects entry edits. A new ArrayList copy supports edits to its own membership. The constructor copies element references into another list structure, so it does not promise independent copies of the element objects.

Rereading the report answers a question about an already produced result. Requesting a fresh source stream starts a new computation that can observe the source at a later time. Confusing those actions can lead to stale assumptions about a report or invalid reuse of a stream.

</details>


## Reflection

Describe a report that needs a stable generated result and a separate editable draft. Explain when you would generate a new report and when you would copy an existing report. Identify one mistake caused by confusing a pending computation with a stored result, and explain how your design would prevent it.


In [ ]:
Your response:

Report and editable draft scenario:
When to regenerate:
When to copy:
Possible confusion and prevention:


Next, you will combine elements into one result and compare safe sequential and parallel computations.


## Supplemental Reading

- [Stream lifecycle and toList](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/util/stream/Stream.html) describes single-use streams, terminal operations, and result-list restrictions.
- [Collection stream sources](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/util/Collection.html#stream()) explains source binding and interference expectations.
- [ArrayList constructors](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/util/ArrayList.html) documents copying collection entries into a new list.
